In [1]:
##DEPENDENCIAS
!pip install scikit-learn pymongo python-dotenv gensim transformers torch plotly

In [2]:
##CONEXION A MONGO
import pandas as pd
import numpy as np
from pymongo import MongoClient
from dotenv import load_dotenv
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, silhouette_score, classification_report
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder
from gensim.models import Word2Vec
from transformers import BertTokenizer, BertModel
import torch
import plotly.express as px
import plotly.graph_objects as go
import re
import nltk
from nltk.corpus import stopwords

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")
client = MongoClient(MONGO_URI)
db = client["proyecto2_musica"]
coleccion = db["canciones"]

print("Conectado a MongoDB Atlas")
print(f"Total canciones: {coleccion.count_documents({})}")

Conectado a MongoDB Atlas
Total canciones: 10903


In [3]:
##EXTRAER DATOS DE MONGO
canciones = list(coleccion.find(
    {
        "letra": {"$exists": True, "$ne": None},
        "genero": {"$exists": True, "$ne": None}
    },
    {"letra": 1, "genero": 1, "artista": 1, "titulo": 1, "_id": 0}
))

df = pd.DataFrame(canciones)
df = df.dropna(subset=["letra", "genero"])

print(f"Canciones cargadas: {len(df)}")
print(f"\nPor genero:")
print(df["genero"].value_counts())

Canciones cargadas: 10000

Por genero:
genero
pop        2494
country    1950
blues      1610
rock       1404
jazz       1328
reggae      893
hip hop     321
Name: count, dtype: int64


In [4]:
##PRE PROCESAMIENTO
nltk.download("stopwords")
stop_es = set(stopwords.words("spanish"))
stop_en = set(stopwords.words("english"))
stopwords_all = stop_es.union(stop_en)

def limpiar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúüñ\s]", "", texto)
    tokens = texto.split()
    tokens = [t for t in tokens if t not in stopwords_all and len(t) > 2]
    return " ".join(tokens)

def tokenizar(texto):
    return limpiar_texto(texto).split()

df["texto_limpio"] = df["letra"].apply(limpiar_texto)
df["tokens"] = df["letra"].apply(tokenizar)
df = df[df["texto_limpio"].apply(len) > 20]

le = LabelEncoder()
df["genero_encoded"] = le.fit_transform(df["genero"])

print(f"Preprocesamiento completo: {len(df)} canciones validas")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\98248\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Preprocesamiento completo: 9991 canciones validas


In [5]:
## BOW/TF-IDF
print("Generando representacion TF-IDF...")

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df["texto_limpio"])
y = df["genero_encoded"]

print(f"Matriz TF-IDF: {X_tfidf.shape}")

Generando representacion TF-IDF...
Matriz TF-IDF: (9991, 5000)


In [6]:
##WORD2VEC
print("Cargando modelo Word2Vec...")

modelo_cbow = Word2Vec.load("../data/processed/w2v_cbow.model")

def vectorizar_w2v(tokens, modelo):
    vectores = [modelo.wv[t] for t in tokens if t in modelo.wv]
    if vectores:
        return np.mean(vectores, axis=0)
    return np.zeros(modelo.vector_size)

print("Generando vectores Word2Vec...")
X_w2v = np.array([vectorizar_w2v(tokens, modelo_cbow) for tokens in df["tokens"]])

print(f"Matriz Word2Vec: {X_w2v.shape}")

Cargando modelo Word2Vec...
Generando vectores Word2Vec...
Matriz Word2Vec: (9991, 100)


In [7]:
##BETO 768
print("Cargando BETO...")

modelo_beto = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = BertTokenizer.from_pretrained(modelo_beto)
model = BertModel.from_pretrained(modelo_beto)
model.eval()

def get_embedding_beto(texto, max_length=512):
    if not isinstance(texto, str) or len(texto.strip()) == 0:
        return np.zeros(768)
    texto = texto[:500]
    inputs = tokenizer(
        texto,
        return_tensors="pt",
        max_length=max_length,
        truncation=True,
        padding=True
    )
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()

print("Generando embeddings BETO (esto tarda varios minutos)...")

# Usar muestra de 1000 canciones para no tardar demasiado
muestra = df.sample(min(1000, len(df)), random_state=42).reset_index(drop=True)
X_beto = np.array([get_embedding_beto(texto) for texto in muestra["texto_limpio"]])
y_muestra = muestra["genero_encoded"].values

print(f"Matriz BETO: {X_beto.shape}")

Cargando BETO...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on 

Generando embeddings BETO (esto tarda varios minutos)...
Matriz BETO: (1000, 768)


In [8]:
##CLASIFICACION POR BETO
from sklearn.model_selection import train_test_split

print("Clasificacion de genero con cada representacion:\n")

resultados = {}

# TF-IDF
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)
clf_tfidf = LogisticRegression(max_iter=1000)
clf_tfidf.fit(X_train, y_train)
acc_tfidf = accuracy_score(y_test, clf_tfidf.predict(X_test))
resultados["TF-IDF"] = acc_tfidf
print(f"TF-IDF accuracy: {acc_tfidf:.4f}")

# Word2Vec
X_train, X_test, y_train, y_test = train_test_split(X_w2v, y, test_size=0.2, random_state=42)
clf_w2v = LogisticRegression(max_iter=1000)
clf_w2v.fit(X_train, y_train)
acc_w2v = accuracy_score(y_test, clf_w2v.predict(X_test))
resultados["Word2Vec"] = acc_w2v
print(f"Word2Vec accuracy: {acc_w2v:.4f}")

# BETO
X_train, X_test, y_train, y_test = train_test_split(X_beto, y_muestra, test_size=0.2, random_state=42)
clf_beto = LogisticRegression(max_iter=1000)
clf_beto.fit(X_train, y_train)
acc_beto = accuracy_score(y_test, clf_beto.predict(X_test))
resultados["BETO"] = acc_beto
print(f"BETO accuracy: {acc_beto:.4f}")

Clasificacion de genero con cada representacion:

TF-IDF accuracy: 0.3637
Word2Vec accuracy: 0.3462
BETO accuracy: 0.2450


C:\Users\98248\Downloads\PYCHAR\PROYECTO_MONGO\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
##CLUSTERING
print("Clustering con KMeans y Silhouette Score:\n")

n_clusters = len(df["genero"].unique())
silhouettes = {}

# TF-IDF
km_tfidf = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels_tfidf = km_tfidf.fit_predict(X_tfidf)
sil_tfidf = silhouette_score(X_tfidf, labels_tfidf, sample_size=2000)
silhouettes["TF-IDF"] = sil_tfidf
print(f"TF-IDF Silhouette Score: {sil_tfidf:.4f}")

# Word2Vec
km_w2v = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels_w2v = km_w2v.fit_predict(X_w2v)
sil_w2v = silhouette_score(X_w2v, labels_w2v)
silhouettes["Word2Vec"] = sil_w2v
print(f"Word2Vec Silhouette Score: {sil_w2v:.4f}")

# BETO
km_beto = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels_beto = km_beto.fit_predict(X_beto)
sil_beto = silhouette_score(X_beto, labels_beto)
silhouettes["BETO"] = sil_beto
print(f"BETO Silhouette Score: {sil_beto:.4f}")

Clustering con KMeans y Silhouette Score:

TF-IDF Silhouette Score: 0.0050
Word2Vec Silhouette Score: 0.0558
BETO Silhouette Score: 0.0263


In [10]:
##COMPARACION
# Grafico de accuracy
fig1 = px.bar(
    x=list(resultados.keys()),
    y=list(resultados.values()),
    title="Comparacion de Accuracy por Representacion",
    labels={"x": "Representacion", "y": "Accuracy"},
    color=list(resultados.keys()),
    width=700,
    height=400
)
fig1.show()

# Grafico de silhouette
fig2 = px.bar(
    x=list(silhouettes.keys()),
    y=list(silhouettes.values()),
    title="Comparacion de Silhouette Score por Representacion",
    labels={"x": "Representacion", "y": "Silhouette Score"},
    color=list(silhouettes.keys()),
    width=700,
    height=400
)
fig2.show()

In [11]:
## TSNE COMPARATIVO
print("Generando t-SNE comparativo...")

muestra_tsne = df.sample(min(500, len(df)), random_state=42).reset_index(drop=True)

# t-SNE TF-IDF
X_tfidf_muestra = tfidf.transform(muestra_tsne["texto_limpio"])
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tfidf_2d = tsne.fit_transform(X_tfidf_muestra.toarray())

fig3 = px.scatter(
    x=tfidf_2d[:, 0], y=tfidf_2d[:, 1],
    color=muestra_tsne["genero"],
    title="t-SNE TF-IDF por genero",
    width=800, height=500
)
fig3.show()

# t-SNE Word2Vec
X_w2v_muestra = np.array([vectorizar_w2v(tokens, modelo_cbow) for tokens in muestra_tsne["tokens"]])
w2v_2d = tsne.fit_transform(X_w2v_muestra)

fig4 = px.scatter(
    x=w2v_2d[:, 0], y=w2v_2d[:, 1],
    color=muestra_tsne["genero"],
    title="t-SNE Word2Vec por genero",
    width=800, height=500
)
fig4.show()

Generando t-SNE comparativo...


In [12]:
##CERRAR CONEXION
client.close()
print("Conexion cerrada")

Conexion cerrada


In [ ]:
##Conclusion
Con corpus pequeño y muchos géneros, TF-IDF captura bien las palabras características de cada género (vocabulario específico), mientras que BETO genera embeddings muy ricos pero un clasificador simple como Logistic Regression no aprovecha bien esas 768 dimensiones con pocos datos. Es un resultado legítimo y documentarlo así demuestra análisis crítico.